In [1]:
import duckdb
import pandas as pd
from pathlib import Path

In [2]:
# Paths
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

con = duckdb.connect()

print("DuckDB version:", duckdb.__duckdb_version__)

DuckDB version: 1.5.3


In [3]:
TABLES = {
    "application_train":    RAW / "application_train.csv",
    "application_test":     RAW / "application_test.csv",
    "bureau":               RAW / "bureau.csv",
    "bureau_balance":       RAW / "bureau_balance.csv",
    "previous_application": RAW / "previous_application.csv",
    "POS_CASH_balance":     RAW / "POS_CASH_balance.csv",
    "installments_payments":RAW / "installments_payments.csv",
    "credit_card_balance":  RAW / "credit_card_balance.csv",
}

In [6]:
for name, path in TABLES.items():
    result = con.execute(f"""
        SELECT COUNT(*) AS rows,
                COUNT(COLUMNS(*)) AS cols
        FROM '{path}'
    """).df()

    rows = result["rows"].iloc[0]
    cols_df = con.execute(f"DESCRIBE SELECT * FROM '{path}'").df()
    n_cols = len(cols_df)

    print(f"{'─'*55}")
    print(f"  {name}")
    print(f"  Rows: {rows:,}   Columns: {n_cols}")
    print(f"  Cols: {', '.join(cols_df['column_name'].tolist()[:6])} ...")

───────────────────────────────────────────────────────
  application_train
  Rows: 307,511   Columns: 122
  Cols: SK_ID_CURR, TARGET, NAME_CONTRACT_TYPE, CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY ...
───────────────────────────────────────────────────────
  application_test
  Rows: 48,744   Columns: 121
  Cols: SK_ID_CURR, NAME_CONTRACT_TYPE, CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY, CNT_CHILDREN ...
───────────────────────────────────────────────────────
  bureau
  Rows: 1,716,428   Columns: 17
  Cols: SK_ID_CURR, SK_ID_BUREAU, CREDIT_ACTIVE, CREDIT_CURRENCY, DAYS_CREDIT, CREDIT_DAY_OVERDUE ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

───────────────────────────────────────────────────────
  bureau_balance
  Rows: 27,299,925   Columns: 3
  Cols: SK_ID_BUREAU, MONTHS_BALANCE, STATUS ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

───────────────────────────────────────────────────────
  previous_application
  Rows: 1,670,214   Columns: 37
  Cols: SK_ID_PREV, SK_ID_CURR, NAME_CONTRACT_TYPE, AMT_ANNUITY, AMT_APPLICATION, AMT_CREDIT ...


...

───────────────────────────────────────────────────────
  POS_CASH_balance
  Rows: 10,001,358   Columns: 8
  Cols: SK_ID_PREV, SK_ID_CURR, MONTHS_BALANCE, CNT_INSTALMENT, CNT_INSTALMENT_FUTURE, NAME_CONTRACT_STATUS ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

───────────────────────────────────────────────────────
  installments_payments
  Rows: 13,605,401   Columns: 8
  Cols: SK_ID_PREV, SK_ID_CURR, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER, DAYS_INSTALMENT, DAYS_ENTRY_PAYMENT ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

───────────────────────────────────────────────────────
  credit_card_balance
  Rows: 3,840,312   Columns: 23
  Cols: SK_ID_PREV, SK_ID_CURR, MONTHS_BALANCE, AMT_BALANCE, AMT_CREDIT_LIMIT_ACTUAL, AMT_DRAWINGS_ATM_CURRENT ...


In [7]:
target_dist = con.execute(f"""
    SELECT TARGET,
           COUNT(*) AS count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM '{RAW}/application_train.csv'
    GROUP BY TARGET
    ORDER BY TARGET
""").df()

print(target_dist)

   TARGET   count    pct
0       0  282686  91.93
1       1   24825   8.07


In [8]:
for name, path in TABLES.items():
    out = PROCESSED / f"{name}.parquet"
    con.execute(f"""
        COPY (SELECT * FROM '{path}')
        TO '{out}'
        (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    print(f"✓  {name}.parquet")

print("\nAll files converted.")

✓  application_train.parquet
✓  application_test.parquet
✓  bureau.parquet
✓  bureau_balance.parquet
✓  previous_application.parquet
✓  POS_CASH_balance.parquet
✓  installments_payments.parquet
✓  credit_card_balance.parquet

All files converted.


In [9]:
print(f"{'Table':<30} {'Parquet size':>14} {'Rows':>12}")
print("─" * 58)
for name in TABLES:
    out = PROCESSED / f"{name}.parquet"
    size_mb = out.stat().st_size / 1_048_576
    rows = con.execute(f"SELECT COUNT(*) FROM '{out}'").fetchone()[0]
    print(f"{name:<30} {size_mb:>11.1f} MB {rows:>12,}")

Table                            Parquet size         Rows
──────────────────────────────────────────────────────────
application_train                     23.5 MB      307,511
application_test                       4.1 MB       48,744
bureau                                35.3 MB    1,716,428
bureau_balance                        16.8 MB   27,299,925
previous_application                  67.8 MB    1,670,214
POS_CASH_balance                     118.0 MB   10,001,358
installments_payments                325.9 MB   13,605,401
credit_card_balance                  127.6 MB    3,840,312
